#### 1. Load Audited Data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/audited_2025.csv')

scorecard = df.groupby(['Supplier', 'Category']).agg(
    Total_Orders=('PO_Number', 'count'),
    Avg_OTD_Rate=('OTD_Status', lambda x: (x == 'On-Time').mean() * 100),
    Avg_Price_Variance=('Price_Variance_Pct', 'mean'),
    Total_Spend=('Unit_Price_Final', 'sum'),
    Max_Delay=('Days_Late', 'max')
).reset_index()

#### 2. The Weighted Scoring Logic

In [2]:
W_RELIABILITY = 0.50 
W_COST_STABILITY = 0.30 
W_VOLUME_BONUS = 0.20 

# 1. Reliability Score
scorecard['Score_OTD'] = scorecard['Avg_OTD_Rate']

# 2. Cost Score
scorecard['Score_Price'] = 100 - scorecard['Avg_Price_Variance'].clip(0, 100)

# 3. Volume Score
scorecard['Score_Volume'] = (np.log1p(scorecard['Total_Orders']) / np.log1p(scorecard['Total_Orders'].max())) * 100

# Final Weighted Score
scorecard['Final_Score'] = (
    (scorecard['Score_OTD'] * W_RELIABILITY) +
    (scorecard['Score_Price'] * W_COST_STABILITY) +
    (scorecard['Score_Volume'] * W_VOLUME_BONUS)
).round(2)

#### 3. The "Tiering" Logic (A-F Grading)

In [3]:
def get_tier(score):
    if score >= 90: return 'A (Strategic Partner)'
    if score >= 80: return 'B (Preferred)'
    if score >= 70: return 'C (Active Monitoring)'
    return 'F (Critical Risk / Re-Bid)'

scorecard['Supplier_Tier'] = scorecard['Final_Score'].apply(get_tier)

scorecard = scorecard.sort_values(by='Final_Score', ascending=False)
print("Top Performers (The 'Partners'):")
print(scorecard[['Supplier', 'Final_Score', 'Supplier_Tier']].head(3))

Top Performers (The 'Partners'):
               Supplier  Final_Score          Supplier_Tier
1      Dayliff (Global)       100.00  A (Strategic Partner)
5      Pedrollo (Italy)        94.56  A (Strategic Partner)
6  SolarWorld (Germany)        93.34  A (Strategic Partner)


#### 4. The "Volatility" Flag

In [5]:
scorecard['Risk_Alert'] = np.where(scorecard['Max_Delay'] > 20, "High Volatility", "Stable")

scorecard.to_csv('../data/processed/final_scorecard_2025.csv', index=False)
print("Executive Scorecard Generated.")

Executive Scorecard Generated.


In [6]:
scorecard.tail()

,Supplier,Category,Total_Orders,Avg_OTD_Rate,Avg_Price_Variance,Total_Spend,Max_Delay,Score_OTD,Score_Price,Score_Volume,Final_Score,Supplier_Tier,Risk_Alert
7,Speroni (Italy),External OEM,26,100.000000,1.923077,21008.049,0,100.000000,98.076923,68.723536,93.17,A (Strategic Partner),Stable
0,Addis Engineering,Local Partner,19,94.736842,0.000000,16822.260,8,94.736842,100.000000,62.465870,89.86,B (Preferred),Stable
3,Grundfos (Denmark),External OEM,37,89.189189,1.621622,33698.554,30,89.189189,98.378378,75.849563,89.28,B (Preferred),High Volatility
2,Ethio-Plastics,Local Partner,23,73.913043,0.000000,18189.780,8,73.913043,100.000000,66.267569,80.21,B (Preferred),Stable
4,Nile Logistics,Local Partner,14,71.428571,0.000000,11607.790,8,71.428571,100.000000,56.467233,77.01,C (Active Monitoring),Stable
